# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Title:** Ranking Content for Refresh Review from Observable Search Signals

**Author:** Yuan Andrei C. Mariano · Computer Science (Data Science)

Repo: https://github.com/yuan05-afk/flyrank-ml-internship


## 1. Question

Which content items should an editor refresh first, given observable search and engagement
signals, so rewrite effort goes to declining or stale-but-visible pages rather than random
or merely high-volume pages?


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

print('Unit: content item | Output: ranked review queue | Action: editor prioritizes refresh')


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32
Unit: content item | Output: ranked review queue | Action: editor prioritizes refresh


## 2. Data

Starter: 30,000 anonymized pages × 44 columns, 32 clients, trailing-90-day metrics.
Excluded: trend_direction/trend_pct as features; all identifying text/URLs (not present).
Warehouse (~79M daily facts) deferred without HF token; SETUP.md has request steps.


In [2]:
print(df.shape)
print('clients', df['client_id'].nunique())


(30000, 44)
clients 32


## 3. Methodology

Label: is_declining_label from observed trend_direction==down.
Features: MODEL_* lists in ml_utils (no leakage fields).
Baseline: transparent visibility×freshness×CTR-gap rule.
Models: LR, DT, RF; client-holdout; select by Precision@50.
Leakage checks: forbidden feature intersection empty.


In [3]:
results = json.loads((ROOT/'outputs'/'model_results.json').read_text())
print(results['split_strategy'], results['best_model']['name'], results['best_model']['selection_metric'])


client_holdout random_forest precision_at_50


## 4. Results (vs baseline)

On client-holdout: RF Precision@50 = 0.74 vs baseline 0.24; ROC-AUC 0.75; base rate 0.54.
Lift is real on this split; still decision-support.


In [4]:
b=results['baseline']; m=results['models']['random_forest']
print(f"P@50 RF {m['precision_at_50']:.2f} vs baseline {b['baseline_precision_at_50']:.2f}")
print(f"ROC-AUC {m['roc_auc']:.3f} | AP {m['average_precision']:.3f} | base rate {results['target_positive_rate']:.3f}")


P@50 RF 0.74 vs baseline 0.24
ROC-AUC 0.750 | AP 0.618 | base rate 0.542


## 5. Limitations

Snapshot label (not a sealed future window from warehouse panel). Starter size. No causal
refresh experiment. Rates are ×100 percents. avg_position=0 is missingness.


In [5]:
print('Limitations logged for the paper Limitations section.')


Limitations logged for the paper Limitations section.


## 6. Ranked recommendations

Editors should start from high-confidence `refresh` / `refresh_and_review_ctr` rows, verify
on-page, and skip no-go cases. Monitor Precision@50 vs baseline after each data refresh.


In [6]:
q = pd.read_csv(ROOT/'outputs'/('refresh_queue.csv' if (ROOT/'outputs'/'refresh_queue.csv').exists() else 'refresh_queue_sample.csv'))
print(q.head(5).to_string(index=False))


 final_rank           content_id         client_id  final_refresh_score best_model_name  best_model_probability  baseline_refresh_score confidence       suggested_action                                                                                                                                                   final_reason_codes  is_declining_label  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  word_count trend_direction competition_level    content_type   main_intent age_tier freshness_tier word_count_tier impression_tier position_tier
          1 content_1f080331fa2b client_3fdba35f04            81.734212   random_forest                0.783472                0.844481       high refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate                   1            12834           6            

## 7. Artifacts the paper embeds

- `outputs/model_results.json`, charts in `outputs/charts/`
- `work/outputs/*.json` receipts
- Deployed paper URL in `submission/paper_url.txt`
- Demo outline + social/employer cuts below (ML-12)


In [7]:
print('Charts:', [p.name for p in (ROOT/'outputs'/'charts').glob('*.svg')])


Charts: ['action_mix.svg', 'confidence_mix.svg', 'top_feature_importance.svg', 'top_reason_codes.svg', 'trend_distribution.svg']


## ML-12 — Tell the Story (demo + cuts)

### 5-minute demo outline
1. (30s) Decision: which page to refresh first — cost of wrong call.
2. (60s) Data contract + leakage rule (no trend_* features).
3. (90s) Baseline vs RF on client-holdout Precision@50 with base rate.
4. (90s) Open top-3 queue rows: action + reason codes; what would make them wrong.
5. (30s) Limits + paper URL.

### Social-post cut
Measured on 30k anonymized pages: a transparent refresh rule hit Precision@50≈0.24;
a Random Forest on the same client-holdout hit ≈0.74 (base rate ≈0.54). Decision-support
queue for editors — not a Google predictor. Paper + repo linked.

### Employer-facing summary
Built a content refresh ranking system on FlyRank's anonymized search starter dataset
(30k pages, client-holdout validation). Compared a transparent baseline to RF/LR/DT;
selected by Precision@50; shipped an editor playbook and a public research page with
honest limitations.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
